# Data Cleaning

This notebook prepares the raw Loan Default dataset for modelling. Steps are applied in strict order: rename columns, drop leaky features, impute missing values, winsorize outliers (in raw scale), then apply log transforms.


In [1]:
import pandas as pd
import numpy as np

# Configuration
NUMERIC_COLS = ["term", "credit_score", "ltv", "dtir1", "loan_amount", "income", "property_value"]
CATEGORICAL_COLS = ["loan_limit", "gender", "approv_in_adv", "loan_type", "loan_purpose",
                    "credit_worthiness", "open_credit", "business_or_commercial", "neg_ammortization",
                    "interest_only", "lump_sum_payment", "construction_type", "occupancy_type",
                    "secured_by", "total_units", "credit_type", "co-applicant_credit_type",
                    "age", "submission_of_application", "region", "security_type"] 
LOG_TRANSFORM_COLS = ["loan_amount", "income", "property_value"]
DROP_COLS = ["rate_of_interest", "interest_rate_spread", "upfront_charges"]

# Winsorize only the features that are log-transformed - bounds must be in raw scale
# before the log transform runs. ltv and dtir1 are NOT log-transformed; they are
# winsorized in feature_engineering.ipynb after the split.
WINSORIZE_BOUNDS = {
    'loan_amount':    (66500,   856500),
    'property_value': (88000,  1808000),
    'income':         (600,     26640),
}


In [2]:
def load_data(path):
    return pd.read_csv(path)

In [3]:
def impute_missing_values(df):
    # For numeric columns, replace missing values with median
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # For categorical columns, replace missing values with mode
    for col in CATEGORICAL_COLS:
        df[col] = df[col].fillna(df[col].mode()[0])

    return df

In [4]:
def winsorize_outliers(df):
    for col, (lower, upper) in WINSORIZE_BOUNDS.items():
        if col in df.columns:
            before_min, before_max = df[col].min(), df[col].max()
            df[col] = df[col].clip(lower=lower, upper=upper)
            after_min, after_max = df[col].min(), df[col].max()
            print(f"{col}: [{before_min:,.2f}, {before_max:,.2f}] -> [{after_min:,.2f}, {after_max:,.2f}]")
    return df


In [5]:
def log_transform(df):
    for col in LOG_TRANSFORM_COLS:
        if col in df.columns:
            df[col] = np.log1p(df[col])
    return df


In [6]:
def clean_data(path):
    df = load_data(path)

    # Rename columns for consistency
    df = df.rename(columns={'ID': 'id', 'Gender': 'gender', 'Credit_Worthiness': 'credit_worthiness', 
                        'Interest_rate_spread': 'interest_rate_spread', 'Upfront_charges': 'upfront_charges', 
                       'Neg_ammortization': 'neg_ammortization', 'Secured_by': 'secured_by', 
                       'Credit_Score': 'credit_score', 'LTV': 'ltv', 'Region': 'region',
                       'Security_Type': 'security_type', 'Status': 'status'})

    # Drop features with high missingness or potential data leakage
    df = df.drop(columns=[col for col in DROP_COLS if col in df.columns])

    df = impute_missing_values(df)
    df = winsorize_outliers(df)  # must run before log_transform - bounds are in raw scale
    df = log_transform(df)
    return df


In [7]:
# Execute cleaning pipeline
input_path = "../data/Loan_Default.csv"
output_path = "../data/cleaned_loan_data.csv"

cleaned_df = clean_data(input_path)
cleaned_df.to_csv(output_path, index=False)

print(f"Cleaned data saved to {output_path}")
print(f"Shape: {cleaned_df.shape}")
print(f"Missing values: {cleaned_df.isnull().sum().sum()}")

loan_amount: [16,500.00, 3,576,500.00] -> [66,500.00, 856,500.00]
property_value: [8,000.00, 16,508,000.00] -> [88,000.00, 1,808,000.00]
income: [0.00, 578,580.00] -> [600.00, 26,640.00]
Cleaned data saved to ../data/cleaned_loan_data.csv
Shape: (148670, 31)
Missing values: 0


In [8]:
import os, json

os.makedirs('../models', exist_ok=True)
bounds_path = '../models/raw_winsorize_bounds.json'

# Save the raw-scale winsorization bounds so the full preprocessing pipeline
# can be reproduced on new data without reading back into this notebook.
# feature_engineering.ipynb saves its own bounds (ltv, dtir1) separately.
with open(bounds_path, 'w') as f:
    json.dump(
        {col: list(bounds) for col, bounds in WINSORIZE_BOUNDS.items()},
        f, indent=2
    )

print(f'Raw winsorization bounds saved to {bounds_path}')
print(json.dumps({col: list(b) for col, b in WINSORIZE_BOUNDS.items()}, indent=2))


Raw winsorization bounds saved to ../models/raw_winsorize_bounds.json
{
  "loan_amount": [
    66500,
    856500
  ],
  "property_value": [
    88000,
    1808000
  ],
  "income": [
    600,
    26640
  ]
}
